# Key-Freq Analysis — comparison across all optimizers

Standalone notebook : load Phase 1 final checkpoints (`model.pt`) for every optimizer config (EGD ×4, Muon ×4, AdamW ×2), compute the per-seed Fourier components of $W_L$, and compare 5 key-freq selection methods :

- **topk k=5** — Nanda's hardcoded reference
- **cum_energy τ=0.85 / 0.90 / 0.95** — keep freqs explaining τ % of $W_L$'s spectral energy (Parseval)
- **permutation** — statistical test vs row-shuffle null of $W_L$ (most rigorous, n_perms=500, α=0.05, FDR)

**No training, no file writes** : pure analysis. Reads `runs/phase2_optim/{name}/seed*/model.pt` only.

**Conclusion drawn at the end** : configs where n_keep differs significantly from 5 → candidates for a Phase 2 v2 with flexible-k.

## 1 — Setup

In [10]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch as t
import importlib, viz_analysis, fourier_metrics, model
importlib.reload(viz_analysis); importlib.reload(fourier_metrics); importlib.reload(model)

from model           import Config
from viz_analysis    import identify_key_freqs
from fourier_metrics import make_fourier_basis

ROOT     = Path.cwd()
RUN_ROOT = ROOT / 'runs' / 'phase2_optim'
SEEDS    = [0, 1, 2, 3, 4]
CONVERGED_THRESH = 0.99

BASE_CONFIG = Config(
    p=113, d_model=128, d_mlp=512, num_heads=4, n_ctx=3,
    act_type='ReLU', frac_train=0.3,
    num_epochs=25_000, seed=0,
)

print(f'Reading from : {RUN_ROOT}')
print(f'Seeds        : {SEEDS}')
print(f'Converged threshold : test_acc ≥ {CONVERGED_THRESH}')

Reading from : /Users/mverest/Desktop/Fourier/Opti_ML/baseline/runs/phase2_optim
Seeds        : [0, 1, 2, 3, 4]
Converged threshold : test_acc ≥ 0.99


## 2 — All optimizer configs (10 total)

In [11]:
CONFIGS = {
    # ── EGD ────────────────────────────────────────────────────────────────
    'egd_m0_fast'   : {'optimizer': 'EGD',   'label': 'EGD m=0 fast'},
    'egd_m0_slow'   : {'optimizer': 'EGD',   'label': 'EGD m=0 slow'},
    'egd_m0.9_fast' : {'optimizer': 'EGD',   'label': 'EGD m=0.9 fast'},
    'egd_m0.9_slow' : {'optimizer': 'EGD',   'label': 'EGD m=0.9 slow'},
    # ── Muon ───────────────────────────────────────────────────────────────
    'muon_m0_fast'    : {'optimizer': 'Muon', 'label': 'Muon m=0 fast'},
    'muon_m0_slow'    : {'optimizer': 'Muon', 'label': 'Muon m=0 slow'},
    'muon_m0.95_fast' : {'optimizer': 'Muon', 'label': 'Muon m=0.95 fast'},
    'muon_m0.95_slow' : {'optimizer': 'Muon', 'label': 'Muon m=0.95 slow'},
    # ── AdamW ──────────────────────────────────────────────────────────────
    'adamw_fast' : {'optimizer': 'AdamW', 'label': 'AdamW fast'},
    'adamw_slow' : {'optimizer': 'AdamW', 'label': 'AdamW slow'},
}

for name, cfg in CONFIGS.items():
    cfg['phase1_dir'] = RUN_ROOT / name

# Sanity check : Phase 1 directories exist ?
available = {n: c for n, c in CONFIGS.items() if c['phase1_dir'].exists()}
missing   = [n for n in CONFIGS if n not in available]
print(f'Found {len(available)}/{len(CONFIGS)} Phase 1 directories.')
if missing:
    print(f'⚠ Missing : {missing}')
CONFIGS = available

Found 10/10 Phase 1 directories.


## 3 — Inline helper : load `W_L` + Fourier components from `model.pt`

In [12]:
_basis    = make_fourier_basis(BASE_CONFIG).cpu()
_cos_rows = _basis[1::2]   # (p//2, p)
_sin_rows = _basis[2::2]   # (p//2, p)

def load_components(save_root, seeds, p):
    """Read per-seed model.pt, compute W_L = W_U[:, :p].T @ W_out + Fourier projections."""
    out = {}
    for s in seeds:
        mp = Path(save_root) / f'seed{s}' / 'model.pt'
        if not mp.exists(): continue
        state = t.load(mp, map_location='cpu')
        if isinstance(state, dict) and 'model_state_dict' in state:
            state = state['model_state_dict']
        def _find(suf):
            for k, v in state.items():
                if k.endswith(suf): return v
            return None
        W_E   = _find('W_E')
        W_U   = _find('W_U')
        W_out = _find('W_out')
        if W_E is None or W_U is None or W_out is None: continue
        W_E_p = W_E[:, :p].float().cpu()
        W_L   = (W_U[:, :p].float().cpu().T) @ W_out.float().cpu()
        out[s] = {
            'we_cos':     (W_E_p @ _cos_rows.T).norm(dim=0).numpy(),
            'we_sin':     (W_E_p @ _sin_rows.T).norm(dim=0).numpy(),
            'wl_cos':     (_cos_rows @ W_L).norm(dim=1).numpy(),
            'wl_sin':     (_sin_rows @ W_L).norm(dim=1).numpy(),
            '_W_L':       W_L.numpy(),
            '_basis_cos': _cos_rows.numpy(),
            '_basis_sin': _sin_rows.numpy(),
        }
    return out

def check_converged(save_root, seeds, thresh):
    """From history.json, return seeds with final test_acc >= thresh."""
    import json
    out = []
    for s in seeds:
        hp = Path(save_root) / f'seed{s}' / 'history.json'
        if not hp.exists(): continue
        h = json.loads(hp.read_text())
        if h.get('test_acc') and h['test_acc'][-1] >= thresh:
            out.append(s)
    return out

print('Helpers defined.')

Helpers defined.


## 4 — Apply 5 selection methods to every config × seed

Only seeds that **converged** (final test_acc ≥ 0.99) are analyzed.

In [13]:
ALL_ROWS = []   # per-seed details for table 1
RESULTS  = {}   # full diagnostics dict per config (for downstream use)

for name, cfg in CONFIGS.items():
    converged = check_converged(cfg['phase1_dir'], SEEDS, CONVERGED_THRESH)
    if len(converged) < 1:
        print(f'  ⚠ {name:18s} : 0 converged seed — skip'); continue
    comp = load_components(cfg['phase1_dir'], converged, p=BASE_CONFIG.p)
    if not comp:
        print(f'  ⚠ {name:18s} : no model.pt found — skip'); continue

    kf_topk = identify_key_freqs(comp, method='topk',       k=5,      verbose=False)
    kf_e85  = identify_key_freqs(comp, method='cum_energy', tau=0.85, verbose=False)
    kf_e90  = identify_key_freqs(comp, method='cum_energy', tau=0.90, verbose=False)
    kf_e95  = identify_key_freqs(comp, method='cum_energy', tau=0.95, verbose=False)
    kf_perm = identify_key_freqs(comp, method='permutation',
                                  n_perms=500, alpha=0.05, correction='fdr',
                                  verbose=False)

    RESULTS[name] = {
        'topk_k5':         kf_topk,
        'cum_t0.85':       kf_e85,
        'cum_t0.90':       kf_e90,
        'cum_t0.95':       kf_e95,
        'permutation_fdr': kf_perm,
    }

    for s in sorted(comp.keys()):
        def _fmt(kfd):
            kf = kfd['per_seed'][s]
            return f'({len(kf):2d}) {kf}'
        ALL_ROWS.append({
            'optimizer':    cfg['optimizer'],
            'config':       name,
            'seed':         s,
            'topk k=5':     _fmt(kf_topk),
            'cum τ=0.85':   _fmt(kf_e85),
            'cum τ=0.90':   _fmt(kf_e90),
            'cum τ=0.95':   _fmt(kf_e95),
            'permutation':  _fmt(kf_perm),
        })
    print(f'  ✓ {name:18s} : {len(comp)} seeds analyzed')

print(f'\n→ Analysis done on {len(RESULTS)} configs.')

  ✓ egd_m0_fast        : 5 seeds analyzed


KeyboardInterrupt: 

## 5 — Table 1 : per-seed details (count + freq list)

In [ ]:
df_kf = pd.DataFrame(ALL_ROWS)
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display(df_kf)

,optimizer,config,seed,topk k=5,cum τ=0.85,cum τ=0.90,cum τ=0.95,permutation
0,EGD,egd_m0_fast,0,"( 5) [14, 35, 40, 52, 55]","(11) [1, 8, 14, 31, 33, 35, 40, 42, 51, 52, 55]","(11) [1, 8, 14, 31, 33, 35, 40, 42, 51, 52, 55]","(13) [1, 6, 8, 14, 24, 31, 33, 35, 40, 42, 51, 52, 55]","(13) [1, 6, 8, 14, 24, 31, 33, 35, 40, 42, 51, 52, 55]"
1,EGD,egd_m0_fast,1,"( 5) [14, 24, 29, 33, 48]","(11) [6, 14, 17, 24, 26, 29, 33, 37, 44, 48, 49]","(12) [6, 14, 17, 24, 26, 28, 29, 33, 37, 44, 48, 49]","(13) [6, 14, 17, 24, 26, 28, 29, 30, 33, 37, 44, 48, 49]","(13) [6, 14, 17, 24, 26, 28, 29, 30, 33, 37, 44, 48, 49]"
2,EGD,egd_m0_fast,2,"( 5) [7, 9, 27, 47, 55]","(10) [3, 7, 9, 15, 22, 26, 27, 45, 47, 55]","(11) [3, 7, 9, 14, 15, 22, 26, 27, 45, 47, 55]","(12) [3, 7, 9, 14, 15, 22, 26, 27, 35, 45, 47, 55]","(12) [3, 7, 9, 14, 15, 22, 26, 27, 35, 45, 47, 55]"
3,EGD,egd_m0_fast,3,"( 5) [19, 25, 29, 53, 55]","(11) [3, 7, 8, 19, 25, 29, 37, 38, 49, 53, 55]","(12) [2, 3, 7, 8, 19, 25, 29, 37, 38, 49, 53, 55]","(13) [2, 3, 7, 8, 13, 19, 25, 29, 37, 38, 49, 53, 55]","(13) [2, 3, 7, 8, 13, 19, 25, 29, 37, 38, 49, 53, 55]"
4,EGD,egd_m0_fast,4,"( 5) [7, 14, 31, 47, 53]","(11) [7, 10, 14, 15, 26, 31, 37, 47, 49, 53, 55]","(11) [7, 10, 14, 15, 26, 31, 37, 47, 49, 53, 55]","(12) [7, 10, 14, 15, 26, 28, 31, 37, 47, 49, 53, 55]","(12) [7, 10, 14, 15, 26, 28, 31, 37, 47, 49, 53, 55]"
5,EGD,egd_m0_slow,0,"( 5) [8, 15, 16, 35, 43]","( 3) [15, 16, 35]","( 4) [8, 15, 16, 35]","( 4) [8, 15, 16, 35]","( 4) [8, 15, 16, 35]"
6,EGD,egd_m0_slow,1,"( 5) [17, 34, 37, 38, 49]","( 5) [17, 34, 37, 38, 49]","( 5) [17, 34, 37, 38, 49]","( 5) [17, 34, 37, 38, 49]","( 5) [17, 34, 37, 38, 49]"
7,EGD,egd_m0_slow,2,"( 5) [6, 26, 28, 37, 47]","( 4) [6, 26, 37, 47]","( 5) [6, 26, 28, 37, 47]","( 5) [6, 26, 28, 37, 47]","( 5) [6, 26, 28, 37, 47]"
8,EGD,egd_m0_slow,3,"( 5) [2, 6, 7, 13, 49]","( 3) [7, 13, 49]","( 4) [2, 7, 13, 49]","( 4) [2, 7, 13, 49]","( 4) [2, 7, 13, 49]"
9,EGD,egd_m0_slow,4,"( 5) [7, 37, 47, 49, 53]","( 4) [7, 37, 47, 49]","( 5) [7, 37, 47, 49, 53]","( 5) [7, 37, 47, 49, 53]","( 5) [7, 37, 47, 49, 53]"


## 6 — Table 2 : `n_keep` summary per config × method (mean ± std, range)

In [ ]:
summary_rows = []
for name, results in RESULTS.items():
    row = {'optimizer': CONFIGS[name]['optimizer'], 'config': name}
    for method_name, kf in results.items():
        counts = [len(v) for v in kf['per_seed'].values()]
        if counts:
            row[method_name] = f'{np.mean(counts):.1f} ± {np.std(counts):.1f}   (range {min(counts)}–{max(counts)})'
    summary_rows.append(row)
df_summary = pd.DataFrame(summary_rows)
with pd.option_context('display.max_colwidth', None):
    display(df_summary)

,optimizer,config,topk_k5,cum_t0.85,cum_t0.90,cum_t0.95,permutation_fdr
0,EGD,egd_m0_fast,5.0 ± 0.0 (range 5–5),10.8 ± 0.4 (range 10–11),11.4 ± 0.5 (range 11–12),12.6 ± 0.5 (range 12–13),12.6 ± 0.5 (range 12–13)
1,EGD,egd_m0_slow,5.0 ± 0.0 (range 5–5),3.8 ± 0.7 (range 3–5),4.6 ± 0.5 (range 4–5),4.6 ± 0.5 (range 4–5),4.6 ± 0.5 (range 4–5)
2,EGD,egd_m0.9_fast,5.0 ± 0.0 (range 5–5),4.6 ± 0.5 (range 4–5),5.0 ± 0.0 (range 5–5),5.0 ± 0.0 (range 5–5),5.0 ± 0.0 (range 5–5)
3,EGD,egd_m0.9_slow,5.0 ± 0.0 (range 5–5),3.8 ± 0.7 (range 3–5),4.0 ± 0.9 (range 3–5),4.2 ± 0.7 (range 3–5),4.2 ± 0.7 (range 3–5)
4,Muon,muon_m0_fast,5.0 ± 0.0 (range 5–5),15.8 ± 0.7 (range 15–17),17.4 ± 0.8 (range 17–19),18.4 ± 0.8 (range 18–20),18.4 ± 0.8 (range 18–20)
5,Muon,muon_m0_slow,5.0 ± 0.0 (range 5–5),4.6 ± 0.5 (range 4–5),4.8 ± 0.4 (range 4–5),5.4 ± 0.5 (range 5–6),5.4 ± 0.5 (range 5–6)
6,Muon,muon_m0.95_fast,5.0 ± 0.0 (range 5–5),4.6 ± 0.8 (range 4–6),5.2 ± 1.2 (range 4–7),5.2 ± 1.2 (range 4–7),5.2 ± 1.2 (range 4–7)
7,Muon,muon_m0.95_slow,5.0 ± 0.0 (range 5–5),4.8 ± 0.7 (range 4–6),5.0 ± 0.9 (range 4–6),5.2 ± 1.2 (range 4–7),5.2 ± 1.2 (range 4–7)
8,AdamW,adamw_fast,5.0 ± 0.0 (range 5–5),3.2 ± 0.4 (range 3–4),3.2 ± 0.4 (range 3–4),3.2 ± 0.4 (range 3–4),3.2 ± 0.4 (range 3–4)
9,AdamW,adamw_slow,5.0 ± 0.0 (range 5–5),4.0 ± 0.6 (range 3–5),4.6 ± 0.5 (range 4–5),4.6 ± 0.5 (range 4–5),4.6 ± 0.5 (range 4–5)


## 7 — Table 3 : compact `n_keep` heatmap (mean only)

Easier to spot which configs deviate from k=5.

In [ ]:
compact_rows = []
for name, results in RESULTS.items():
    row = {'optimizer': CONFIGS[name]['optimizer'], 'config': name}
    for method_name, kf in results.items():
        counts = [len(v) for v in kf['per_seed'].values()]
        row[method_name] = round(np.mean(counts), 1) if counts else np.nan
    compact_rows.append(row)
df_compact = pd.DataFrame(compact_rows).set_index(['optimizer', 'config'])

# Highlight cells deviating from 5 (or from topk k=5 reference column)
def _color(v):
    if pd.isna(v): return ''
    if abs(v - 5) >= 2:  return 'background-color: #ffcccc'   # red : far from 5
    if abs(v - 5) >= 1:  return 'background-color: #fff2cc'   # yellow : slightly off
    return ''

display(df_compact.style.applymap(_color)
         .format('{:.1f}')
         .set_caption('Mean n_keep per config × method  ·  red = |Δ| ≥ 2 vs k=5  ·  yellow = |Δ| ≥ 1'))

/var/folders/t_/qjzw0tfs4c1b4y9_lbr0j6x00000gn/T/ipykernel_75393/1881658646.py:17: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  display(df_compact.style.applymap(_color)


## 8 — Conclusion : candidates for Phase 2 v2 with flexible-k

Configs where the cum_energy / permutation methods disagree significantly with k=5 are the ones where the topk-5 selection is sub-optimal. These are the priority candidates for a re-train with `fixed_key_freqs` updated.

In [ ]:
REFERENCE_METHOD = 'cum_t0.90'   # ← change this to 'permutation_fdr' for max rigor
DELTA_THRESH    = 1.0            # ← flag configs where |mean n_keep - 5| ≥ this

candidates = []
for name, results in RESULTS.items():
    counts = [len(v) for v in results[REFERENCE_METHOD]['per_seed'].values()]
    if not counts: continue
    mean_n = np.mean(counts)
    delta  = mean_n - 5
    if abs(delta) >= DELTA_THRESH:
        candidates.append({
            'optimizer':         CONFIGS[name]['optimizer'],
            'config':            name,
            f'mean n_keep ({REFERENCE_METHOD})': round(mean_n, 1),
            'Δ vs k=5':          f'{delta:+.1f}',
            'range':             f'{min(counts)}–{max(counts)}',
            'verdict':           'TOO SPARSE (k=5 overshoots)' if delta < 0 else 'TOO DIFFUSE (k=5 under-counts)',
        })

if candidates:
    print(f'═══ Candidates for Phase 2 v2 (reference method = {REFERENCE_METHOD}, |Δ| ≥ {DELTA_THRESH}) ═══')
    display(pd.DataFrame(candidates))
else:
    print(f'No config deviates by ≥ {DELTA_THRESH} from k=5 using {REFERENCE_METHOD}.')
    print('→ The k=5 selection is consistent with the data ; no Phase 2 v2 needed.')

═══ Candidates for Phase 2 v2 (reference method = cum_t0.90, |Δ| ≥ 1.0) ═══


,optimizer,config,mean n_keep (cum_t0.90),Δ vs k=5,range,verdict
0,EGD,egd_m0_fast,11.4,+6.4,11–12,TOO DIFFUSE (k=5 under-counts)
1,EGD,egd_m0.9_slow,4.0,-1.0,3–5,TOO SPARSE (k=5 overshoots)
2,Muon,muon_m0_fast,17.4,+12.4,17–19,TOO DIFFUSE (k=5 under-counts)
3,AdamW,adamw_fast,3.2,-1.8,3–4,TOO SPARSE (k=5 overshoots)


## 9 — Export full per-seed key freqs (for a future Phase 2 v2)

In [ ]:
# Build a dict { config : { method : { seed : [freqs] } } } ready to drop into Phase 2 v2
EXPORT = {
    name: {method: kf['per_seed'] for method, kf in results.items()}
    for name, results in RESULTS.items()
}

# Quick peek at one config
if EXPORT:
    first = next(iter(EXPORT))
    print(f'Example — {first} :')
    for method, per_seed in EXPORT[first].items():
        print(f'  {method:18s} : {per_seed}')

print(f'\nFull export available in variable `EXPORT` ({len(EXPORT)} configs × 5 methods).')

Example — egd_m0_fast :
  topk_k5            : {0: [14, 35, 40, 52, 55], 1: [14, 24, 29, 33, 48], 2: [7, 9, 27, 47, 55], 3: [19, 25, 29, 53, 55], 4: [7, 14, 31, 47, 53]}
  cum_t0.85          : {0: [1, 8, 14, 31, 33, 35, 40, 42, 51, 52, 55], 1: [6, 14, 17, 24, 26, 29, 33, 37, 44, 48, 49], 2: [3, 7, 9, 15, 22, 26, 27, 45, 47, 55], 3: [3, 7, 8, 19, 25, 29, 37, 38, 49, 53, 55], 4: [7, 10, 14, 15, 26, 31, 37, 47, 49, 53, 55]}
  cum_t0.90          : {0: [1, 8, 14, 31, 33, 35, 40, 42, 51, 52, 55], 1: [6, 14, 17, 24, 26, 28, 29, 33, 37, 44, 48, 49], 2: [3, 7, 9, 14, 15, 22, 26, 27, 45, 47, 55], 3: [2, 3, 7, 8, 19, 25, 29, 37, 38, 49, 53, 55], 4: [7, 10, 14, 15, 26, 31, 37, 47, 49, 53, 55]}
  cum_t0.95          : {0: [1, 6, 8, 14, 24, 31, 33, 35, 40, 42, 51, 52, 55], 1: [6, 14, 17, 24, 26, 28, 29, 30, 33, 37, 44, 48, 49], 2: [3, 7, 9, 14, 15, 22, 26, 27, 35, 45, 47, 55], 3: [2, 3, 7, 8, 13, 19, 25, 29, 37, 38, 49, 53, 55], 4: [7, 10, 14, 15, 26, 28, 31, 37, 47, 49, 53, 55]}
  permutation_fdr   